In [1]:
import pandas as pd 
from pathlib import Path
AI_EXPOSURE_PATH = Path("../data/ai_exposure_results.csv")

if not AI_EXPOSURE_PATH.exists():
    raise FileNotFoundError(f"Exposure results not found : {AI_EXPOSURE_PATH.resolve()}")

ai_exposure_df = pd.read_csv(AI_EXPOSURE_PATH)

print("Skills analyzed :" , ai_exposure_df["skill"].nunique())
print("\nClassification distribution :")
print(ai_exposure_df["overall_classification"].value_counts(normalize=True).sort_index())

print("\nExposure score summary:")
print(ai_exposure_df["ai_exposure_score"].describe())

display(
    ai_exposure_df[
        ["skill", "overall_classification", "ai_exposure_score"]
    ].head(10)
)

Skills analyzed : 60

Classification distribution :
overall_classification
AI-augmented        0.950000
AI-complementary    0.033333
AI-substitutable    0.016667
Name: proportion, dtype: float64

Exposure score summary:
count    60.000000
mean      0.636667
std       0.108872
min       0.200000
25%       0.650000
50%       0.650000
75%       0.650000
max       0.850000
Name: ai_exposure_score, dtype: float64


,skill,overall_classification,ai_exposure_score
0,javascript,AI-augmented,0.65
1,communication,AI-augmented,0.40
2,css,AI-augmented,0.65
3,mysql,AI-augmented,0.75
4,html,AI-augmented,0.75
5,object-oriented programming,AI-augmented,0.65
6,collaboration,AI-complementary,0.30
7,php,AI-augmented,0.65
8,postgresql,AI-augmented,0.65
9,python,AI-augmented,0.65


In [2]:
# 1. Check score diversity
print("Unique exposure scores:",
      ai_exposure_df["ai_exposure_score"].nunique())

print("\nMost frequent scores:")
display(
    ai_exposure_df["ai_exposure_score"]
    .value_counts()
    .sort_values(ascending=False)
    .rename_axis("exposure_score")
    .reset_index(name="skill_count")
)

# 2. Inspect skills with the lowest and highest exposure
print("\nLowest exposure skills:")
display(
    ai_exposure_df
    .sort_values("ai_exposure_score")
    [["skill", "overall_classification", "ai_exposure_score"]]
    .head(10)
)

print("\nHighest exposure skills:")
display(
    ai_exposure_df
    .sort_values("ai_exposure_score", ascending=False)
    [["skill", "overall_classification", "ai_exposure_score"]]
    .head(10)
)

# 3. Compare classifications against numeric scores
print("\nClassification vs. exposure score:")
display(
    ai_exposure_df.groupby("overall_classification")
    ["ai_exposure_score"]
    .agg(["count", "mean", "min", "max"])
    .round(3)
)

Unique exposure scores: 10

Most frequent scores:


,exposure_score,skill_count
0,0.65,38
1,0.75,8
2,0.70,4
3,0.40,3
4,0.60,2
5,0.30,1
6,0.20,1
7,0.50,1
8,0.45,1
9,0.85,1



Lowest exposure skills:


,skill,overall_classification,ai_exposure_score
17,mentoring,AI-complementary,0.20
6,collaboration,AI-complementary,0.30
12,agile,AI-augmented,0.40
1,communication,AI-augmented,0.40
39,github,AI-augmented,0.40
24,encryption,AI-augmented,0.45
22,presentation skills,AI-augmented,0.50
25,penetration testing,AI-augmented,0.60
31,git,AI-augmented,0.60
11,express.js,AI-augmented,0.65



Highest exposure skills:


,skill,overall_classification,ai_exposure_score
59,jquery,AI-substitutable,0.85
3,mysql,AI-augmented,0.75
34,artificial intelligence,AI-augmented,0.75
4,html,AI-augmented,0.75
35,natural language processing,AI-augmented,0.75
44,pandas,AI-augmented,0.75
36,machine learning,AI-augmented,0.75
40,r,AI-augmented,0.75
33,shell scripting,AI-augmented,0.75
28,test-driven development,AI-augmented,0.70



Classification vs. exposure score:


,count,mean,min,max
overall_classification,,,,
AI-augmented,57,0.646,0.40,0.75
AI-complementary,2,0.250,0.20,0.30
AI-substitutable,1,0.850,0.85,0.85


In [3]:
import json
import pandas as pd

skills_to_review = [
    "python",
    "communication",
    "mentoring",
    "jquery",
    "collaboration"
]

for skill in skills_to_review:
    rows = ai_exposure_df[
        ai_exposure_df["skill"].str.lower() == skill
    ]

    if rows.empty:
        print(f"\n{skill}: Not analyzed yet")
        continue

    row = rows.iloc[0]

    print("\n" + "=" * 70)
    print(f"SKILL: {row['skill']}")
    print(f"CLASSIFICATION: {row['overall_classification']}")
    print(f"EXPOSURE SCORE: {row['ai_exposure_score']}")

    tasks = json.loads(row["tasks"])

    for i, task in enumerate(tasks, start=1):
        print(f"\nTask {i}: {task.get('task', 'N/A')}")
        print(f"Classification: {task.get('classification', 'N/A')}")
        print(f"Reason: {task.get('reason', 'N/A')}")

    print(f"\nSummary: {row['summary']}")


SKILL: python
CLASSIFICATION: AI-augmented
EXPOSURE SCORE: 0.65

Task 1: Automated data processing and workflow implementation
Classification: AI-substitutable
Reason: Python scripts for data pipelines and structured/unstructured dataset processing are increasingly generated or optimized by AI tools with minimal manual intervention.

Task 2: Writing scripts for configuration compliance and system scanning
Classification: AI-substitutable
Reason: Standardized scripting for checking baseline deviations and system configuration is a high-volume, repeatable task well-suited for AI generation.

Task 3: Microservice-based infrastructure development
Classification: AI-augmented
Reason: While AI can generate boilerplate code for microservices, the architecture design, integration, and security logic require human verification and domain-specific judgment.

Task 4: Collaborative problem-solving and cross-functional system design
Classification: AI-complementary
Reason: These roles require deep

In [5]:
import json
import pandas as pd

# Provisional PathForge task-exposure weights
TASK_EXPOSURE_WEIGHTS = {
    "AI-substitutable": 1.00,
    "AI-augmented": 0.60,
    "AI-complementary": 0.20
}

def calculate_task_based_exposure(tasks):
    if not tasks:
        return None

    weights = []

    for task in tasks:
        classification = task.get("classification")

        if classification not in TASK_EXPOSURE_WEIGHTS:
            raise ValueError(
                f"Unknown task classification: {classification}"
            )

        weights.append(TASK_EXPOSURE_WEIGHTS[classification])

    return sum(weights) / len(weights)


# Recalculate all currently analyzed skills
recalculated_scores = []

for _, row in ai_exposure_df.iterrows():
    tasks = json.loads(row["tasks"])

    score = calculate_task_based_exposure(tasks)

    recalculated_scores.append({
        "skill": row["skill"],
        "gemini_score": row["ai_exposure_score"],
        "task_based_score": score,
        "task_count": len(tasks)
    })

task_exposure_df = pd.DataFrame(recalculated_scores)

task_exposure_df["score_difference"] = (
    abs(task_exposure_df["task_based_score"]
    - task_exposure_df["gemini_score"])
)

display(
    task_exposure_df.sort_values(
        "score_difference",
        key=lambda x: x.abs(),
        ascending=False
    ).head(15)
)

print("\nScore comparison:")
display(
    task_exposure_df[
        ["gemini_score", "task_based_score"]
    ].describe().round(3)
)

,skill,gemini_score,task_based_score,task_count,score_difference
7,php,0.65,0.400000,4,0.250000
44,pandas,0.75,0.500000,4,0.250000
13,product management,0.65,0.440000,5,0.210000
17,mentoring,0.20,0.400000,4,0.200000
45,numpy,0.65,0.466667,3,0.183333
52,apache,0.65,0.466667,3,0.183333
49,docker,0.65,0.500000,4,0.150000
23,kafka,0.65,0.500000,4,0.150000
26,infrastructure as code,0.65,0.500000,4,0.150000
55,data visualization,0.65,0.500000,4,0.150000



Score comparison:


,gemini_score,task_based_score
count,60.000,60.000
mean,0.637,0.572
std,0.109,0.093
min,0.200,0.360
25%,0.650,0.500
50%,0.650,0.600
75%,0.650,0.600
max,0.850,0.733


In [6]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data")

req_skill_graph_df = pd.read_csv(
    DATA_DIR / "req_skill_graph.csv"
)

print("Shape:", req_skill_graph_df.shape)
print("\nColumns:")
print(req_skill_graph_df.columns.tolist())

print("\nFirst 5 rows:")
display(req_skill_graph_df.head())

Shape: (4692, 7)

Columns:
['Unnamed: 0', 'role', 'skill', 'job_count', 'role_job_count', 'skill_demand_pct', 'requirement_level']

First 5 rows:


,Unnamed: 0,role,skill,job_count,role_job_count,skill_demand_pct,requirement_level
0,963,Android Developer,java,12,16,75.00,Core
1,964,Android Developer,kotlin,10,16,62.50,Core
2,965,Android Developer,android development,7,16,43.75,Common
3,966,Android Developer,agile,6,16,37.50,Common
4,967,Android Developer,communication,5,16,31.25,Common


In [7]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data")

reliable_role_skill_df = pd.read_csv(
    DATA_DIR / "req_skill_graph.csv"
)

# Remove the unnecessary saved index, if present
reliable_role_skill_df = reliable_role_skill_df.drop(
    columns=["Unnamed: 0"],
    errors="ignore"
)

# Normalize skill names for consistent merging
reliable_role_skill_df["skill"] = (
    reliable_role_skill_df["skill"]
    .astype(str)
    .str.lower()
    .str.strip()
)

print("Reliable role-skill records:", len(reliable_role_skill_df))
print("Unique roles:", reliable_role_skill_df["role"].nunique())
print("Unique skills:", reliable_role_skill_df["skill"].nunique())

print("\nRequirement-level distribution:")
print(reliable_role_skill_df["requirement_level"].value_counts())

display(reliable_role_skill_df.head())

Reliable role-skill records: 4692
Unique roles: 135
Unique skills: 235

Requirement-level distribution:
requirement_level
Supporting    2039
Common        1103
Rare          1052
Core           498
Name: count, dtype: int64


,role,skill,job_count,role_job_count,skill_demand_pct,requirement_level
0,Android Developer,java,12,16,75.00,Core
1,Android Developer,kotlin,10,16,62.50,Core
2,Android Developer,android development,7,16,43.75,Common
3,Android Developer,agile,6,16,37.50,Common
4,Android Developer,communication,5,16,31.25,Common


In [8]:
import pandas as pd
import numpy as np
skill_demand = pd.read_csv("../data/skill_demand.csv")
# 1. Prepare market demand data
market_df = skill_demand[
    ["skill", "job_count", "demand_pct", "category"]
].copy()

market_df["skill"] = market_df["skill"].str.lower().str.strip()

# 2. Prepare transferability data
transfer_df = (
    reliable_role_skill_df.assign(
        skill=reliable_role_skill_df["skill"].str.lower().str.strip()
    )
    .groupby("skill")["role"]
    .nunique()
    .reset_index(name="role_count")
)

total_roles = reliable_role_skill_df["role"].nunique()

transfer_df["transferability_score"] = (
    transfer_df["role_count"] / total_roles
)

# 3. Prepare task-based AI exposure
exposure_df = task_exposure_df[
    ["skill", "task_based_score", "task_count"]
].copy()

exposure_df["skill"] = exposure_df["skill"].str.lower().str.strip()

# 4. Merge all three signals
career_resilience_df = (
    market_df
    .merge(transfer_df, on="skill", how="left")
    .merge(exposure_df, on="skill", how="left")
)

# 5. Normalize market demand to a 0–1 scale
max_demand = career_resilience_df["demand_pct"].max()

career_resilience_df["market_demand_score"] = (
    career_resilience_df["demand_pct"] / max_demand
    if max_demand > 0
    else 0
)

# 6. Preserve missing exposure scores
career_resilience_df["exposure_status"] = np.where(
    career_resilience_df["task_based_score"].notna(),
    "Available - preliminary",
    "Pending analysis"
)

# 7. Calculate a provisional score only when exposure exists
# Higher demand + broader transferability + lower AI exposure
career_resilience_df["resilience_score"] = np.nan

valid = career_resilience_df["task_based_score"].notna()

career_resilience_df.loc[valid, "resilience_score"] = (
    0.40 * career_resilience_df.loc[valid, "market_demand_score"]
    + 0.30 * career_resilience_df.loc[valid, "transferability_score"]
    + 0.30 * (
        1 - career_resilience_df.loc[valid, "task_based_score"]
    )
) * 100

# 8. Inspect the results
career_resilience_df = career_resilience_df.sort_values(
    "resilience_score", ascending=False, na_position="last"
).reset_index(drop=True)

print("Total skills:", len(career_resilience_df))
print(
    "Skills with preliminary resilience scores:",
    career_resilience_df["resilience_score"].notna().sum()
)
print(
    "Skills pending exposure analysis:",
    career_resilience_df["resilience_score"].isna().sum()
)

display(
    career_resilience_df[
        [
            "skill",
            "category",
            "demand_pct",
            "transferability_score",
            "task_based_score",
            "resilience_score",
            "exposure_status"
        ]
    ].head(20)
)

Total skills: 240
Skills with preliminary resilience scores: 60
Skills pending exposure analysis: 180


,skill,category,demand_pct,transferability_score,task_based_score,resilience_score,exposure_status
0,communication,Soft Skills,44.520006,0.903704,0.360000,86.311111,Available - preliminary
1,python,Programming Languages,39.095366,0.814815,0.700000,68.570555,Available - preliminary
2,sql,Programming Languages,32.990669,0.718519,0.600000,63.196763,Available - preliminary
3,agile,Project Management & Methodologies,26.063577,0.725926,0.400000,63.195185,Available - preliminary
4,collaboration,Soft Skills,20.306816,0.733333,0.400000,58.245115,Available - preliminary
5,java,Programming Languages,21.840898,0.651852,0.520000,53.579001,Available - preliminary
6,c,Programming Languages,22.995414,0.555556,0.600000,49.327413,Available - preliminary
7,javascript,Programming Languages,16.179029,0.474074,0.600000,40.758634,Available - preliminary
8,docker,DevOps & CI/CD,10.548790,0.466667,0.500000,38.477798,Available - preliminary
9,machine learning,Data Science & Machine Learning,13.822553,0.437037,0.600000,37.530294,Available - preliminary


In [9]:
skills_dict = {
    "Programming Languages": [
        "python", "java", "c", "c++", "c#", "javascript", "typescript", "go",
        "rust", "ruby", "php", "swift", "kotlin", "scala", "r", "matlab",
        "perl", "dart", "lua", "haskell", "julia", "objective-c", "sql",
        "bash", "shell scripting", "powershell", "assembly", "cobol",
        "fortran", "visual basic",
    ],
    "Web Development - Frontend": [
        "html", "css", "sass", "less", "react", "angular", "vue.js",
        "svelte", "next.js", "nuxt.js", "jquery", "bootstrap",
        "tailwind css", "webpack", "redux", "ember.js", "backbone.js",
    ],
    "Web Development - Backend": [
        "node.js", "express.js", "django", "flask", "fastapi",
        "spring boot", "ruby on rails", "laravel", "asp.net", "nestjs",
        "graphql", "rest api", "soap", "microservices", "grpc",
    ],
    "Databases": [
        "mysql", "postgresql", "sqlite", "mongodb", "redis", "cassandra",
        "oracle", "microsoft sql server", "dynamodb", "elasticsearch",
        "neo4j", "firebase", "mariadb", "couchdb", "snowflake", "bigquery",
    ],
    "Data Science & Machine Learning": [
        "machine learning", "deep learning", "artificial intelligence",
        "natural language processing", "computer vision", "tensorflow",
        "pytorch", "keras", "scikit-learn", "pandas", "numpy",
        "matplotlib", "seaborn", "opencv", "xgboost", "lightgbm", "nltk",
        "spacy", "hugging face transformers", "reinforcement learning",
        "statistics", "data visualization", "feature engineering",
        "model deployment", "mlops",
    ],
    "Data Engineering & Big Data": [
        "apache spark", "hadoop", "kafka", "airflow", "etl",
        "data warehousing", "data pipelines", "dbt", "hive", "flink",
        "presto", "databricks",
    ],
    "Cloud Platforms": [
        "amazon web services", "microsoft azure", "google cloud platform",
        "ibm cloud", "oracle cloud", "heroku", "digitalocean", "cloudflare",
    ],
    "DevOps & CI/CD": [
        "docker", "kubernetes", "jenkins", "github actions",
        "gitlab ci/cd", "terraform", "ansible", "puppet", "chef",
        "ci/cd", "infrastructure as code", "prometheus", "grafana",
        "nginx", "apache",
    ],
    "Mobile Development": [
        "android development", "ios development", "react native",
        "flutter", "xamarin", "ionic",
    ],
    "Testing & QA": [
        "unit testing", "integration testing", "selenium", "pytest",
        "junit", "test-driven development", "cypress", "postman",
        "load testing", "manual testing", "automation testing",
        "qa methodologies",
    ],
    "Version Control & Collaboration": [
        "git", "github", "gitlab", "bitbucket", "svn", "jira",
        "confluence", "trello", "slack", "asana",
    ],
    "Operating Systems": [
        "linux", "windows", "macos", "unix", "ubuntu", "red hat",
    ],
    "Cybersecurity": [
        "network security", "penetration testing", "ethical hacking",
        "encryption", "siem", "firewalls", "vulnerability assessment",
        "owasp", "iso 27001", "security auditing",
    ],
    "Networking": [
        "tcp/ip", "dns", "vpn", "load balancing", "network administration",
        "routing and switching", "http/https",
    ],
    "Design & UI/UX": [
        "figma", "adobe xd", "sketch", "photoshop", "illustrator",
        "ui design", "ux research", "wireframing", "prototyping",
        "user testing",
    ],
    "Project Management & Methodologies": [
        "agile", "scrum", "kanban", "waterfall", "project planning",
        "risk management", "pmp", "six sigma", "product management",
    ],
    "Business & Analytics Tools": [
        "excel", "power bi", "tableau", "google analytics", "sap",
        "salesforce", "looker", "sql server reporting services",
    ],
    "Game Development": [
        "unity", "unreal engine", "game design", "3d modeling", "blender",
        "opengl", "directx",
    ],
    "Embedded Systems & Hardware": [
        "embedded c", "microcontrollers", "arduino", "raspberry pi",
        "rtos", "fpga", "vhdl", "verilog", "iot",
    ],
    "API & Architecture": [
        "rest api", "graphql", "microservices architecture",
        "system design", "design patterns",
        "object-oriented programming", "data structures and algorithms",
    ],
    "Soft Skills": [
        "communication", "teamwork", "leadership", "problem solving",
        "critical thinking", "time management", "adaptability",
        "collaboration", "conflict resolution", "mentoring",
        "presentation skills", "decision making",
    ],
}

In [10]:
import re
flat_skills = [
    (category, skill)
    for category, skills in skills_dict.items()
    for skill in skills
]

def extract_skills_fast(text):
    if pd.isna(text):
        return {}

    text = str(text).lower()
    result = {}

    for category, skill in flat_skills:
        pattern = r"\b" + re.escape(skill.lower()) + r"\b"

        if re.search(pattern, text):
            result.setdefault(category, []).append(skill.lower())

    return result

In [11]:
test_resume = """
Computer Science student with experience in Python, C++, JavaScript,
React, SQL, MongoDB, Git, machine learning and TensorFlow.
Built machine learning projects and web applications.
"""

resume_skills = extract_skills_fast(test_resume)

current_skills = {
    skill.lower().strip()
    for skills in resume_skills.values()
    for skill in skills
}

print("Current skills:", sorted(current_skills))

Current skills: ['c', 'git', 'javascript', 'machine learning', 'mongodb', 'python', 'react', 'sql', 'tensorflow']


In [12]:
# Choose your target role
target_role = "Data Architect"

# Get the role's requirements
target_requirements = reliable_role_skill_df[
    reliable_role_skill_df["role"].str.lower() == target_role.lower()
].copy()

if target_requirements.empty:
    raise ValueError(f"Role not found: {target_role}")

# Identify the user's current skills
current_skills = {
    skill.lower().strip()
    for skills in resume_skills.values()
    for skill in skills
}

# Keep only missing skills
target_requirements["skill"] = (
    target_requirements["skill"].str.lower().str.strip()
)

missing_skills_df = target_requirements[
    ~target_requirements["skill"].isin(current_skills)
].copy()

# Attach market demand, transferability and resilience data
priority_df = missing_skills_df.merge(
    career_resilience_df[
        [
            "skill",
            "market_demand_score",
            "transferability_score",
            "task_based_score",
            "resilience_score",
            "exposure_status"
        ]
    ],
    on="skill",
    how="left"
)

# Prioritize role importance first, then market demand and transferability.
# Missing exposure remains missing; it is not imputed.
priority_df["role_priority_score"] = (
    0.50 * priority_df["skill_demand_pct"].div(100)
    + 0.25 * priority_df["market_demand_score"].fillna(0)
    + 0.25 * priority_df["transferability_score"].fillna(0)
)

priority_df = priority_df.sort_values(
    ["role_priority_score", "skill_demand_pct"],
    ascending=False
).reset_index(drop=True)

display(
    priority_df[
        [
            "skill",
            "requirement_level",
            "skill_demand_pct",
            "market_demand_score",
            "transferability_score",
            "task_based_score",
            "role_priority_score",
            "exposure_status"
        ]
    ].head(20)
)

,skill,requirement_level,skill_demand_pct,market_demand_score,transferability_score,task_based_score,role_priority_score,exposure_status
0,communication,Common,39.285714,1.000000,0.903704,0.36,0.672354,Available - preliminary
1,agile,Common,25.000000,0.585435,0.725926,0.40,0.452840,Available - preliminary
2,collaboration,Common,25.000000,0.456128,0.733333,0.40,0.422365,Available - preliminary
3,leadership,Common,25.000000,0.479218,0.622222,NaN,0.400360,Pending analysis
4,etl,Core,53.571429,0.154885,0.325926,0.60,0.388060,Available - preliminary
5,java,Rare,3.571429,0.490586,0.651852,0.52,0.303467,Available - preliminary
6,snowflake,Common,39.285714,0.090231,0.222222,0.60,0.274542,Available - preliminary
7,data warehousing,Common,35.714286,0.075666,0.192593,0.68,0.245636,Available - preliminary
8,go,Supporting,10.714286,0.250799,0.503704,NaN,0.242197,Pending analysis
9,ci/cd,Rare,3.571429,0.267140,0.525926,0.68,0.216124,Available - preliminary


In [13]:
from pathlib import Path

OUTPUT_DIR = Path("../data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_path = OUTPUT_DIR / "role_skill_priorities.csv"

priority_df.to_csv(output_path, index=False)

print(f"Saved {len(priority_df)} role-specific skill priorities.")
print(f"File: {output_path.resolve()}")

Saved 55 role-specific skill priorities.
File: C:\Users\murli mishra\OneDrive\Desktop\PYTHON\PathForgeAI\data\role_skill_priorities.csv


In [14]:
import pandas as pd

# Work on a copy of the existing priority ranking
roadmap_df = priority_df.copy()

# Define requirement importance
requirement_weights = {
    "Core": 4,
    "Common": 3,
    "Supporting": 2,
    "Rare": 1
}

roadmap_df["requirement_weight"] = (
    roadmap_df["requirement_level"].map(requirement_weights).fillna(0)
)

# Rank Core requirements first, then use the role priority score
roadmap_df = roadmap_df.sort_values(
    ["requirement_weight", "role_priority_score", "skill_demand_pct"],
    ascending=False
).reset_index(drop=True)

# Assign learning priority
roadmap_df["priority"] = range(1, len(roadmap_df) + 1)

# Divide the missing skills into learning phases
def assign_phase(requirement_level):
    if requirement_level == "Core":
        return "Phase 1 - Core Skills"
    elif requirement_level == "Common":
        return "Phase 2 - Important Skills"
    elif requirement_level == "Supporting":
        return "Phase 3 - Supporting Skills"
    else:
        return "Phase 4 - Optional Skills"

roadmap_df["learning_phase"] = (
    roadmap_df["requirement_level"].apply(assign_phase)
)

# Select useful roadmap fields
learning_roadmap = roadmap_df[
    [
        "priority",
        "skill",
        "learning_phase",
        "requirement_level",
        "skill_demand_pct",
        "role_priority_score",
        "task_based_score",
        "exposure_status"
    ]
].copy()

display(learning_roadmap.head(25))

print("\nSkills in roadmap:", len(learning_roadmap))
print("\nSkills per phase:")
print(learning_roadmap["learning_phase"].value_counts())

,priority,skill,learning_phase,requirement_level,skill_demand_pct,role_priority_score,task_based_score,exposure_status
0,1,etl,Phase 1 - Core Skills,Core,53.571429,0.388060,0.600000,Available - preliminary
1,2,communication,Phase 2 - Important Skills,Common,39.285714,0.672354,0.360000,Available - preliminary
2,3,agile,Phase 2 - Important Skills,Common,25.000000,0.452840,0.400000,Available - preliminary
3,4,collaboration,Phase 2 - Important Skills,Common,25.000000,0.422365,0.400000,Available - preliminary
4,5,leadership,Phase 2 - Important Skills,Common,25.000000,0.400360,NaN,Pending analysis
5,6,snowflake,Phase 2 - Important Skills,Common,39.285714,0.274542,0.600000,Available - preliminary
6,7,data warehousing,Phase 2 - Important Skills,Common,35.714286,0.245636,0.680000,Available - preliminary
7,8,databricks,Phase 2 - Important Skills,Common,28.571429,0.215375,0.520000,Available - preliminary
8,9,data pipelines,Phase 2 - Important Skills,Common,25.000000,0.214899,NaN,Pending analysis
9,10,go,Phase 3 - Supporting Skills,Supporting,10.714286,0.242197,NaN,Pending analysis



Skills in roadmap: 55

Skills per phase:
learning_phase
Phase 4 - Optional Skills      28
Phase 3 - Supporting Skills    18
Phase 2 - Important Skills      8
Phase 1 - Core Skills           1
Name: count, dtype: int64


In [15]:
print("Target role:", target_role)

print("\nRequirement levels in target role:")
print(
    target_requirements["requirement_level"]
    .value_counts()
)

print("\nTop 15 missing skills by role demand:")
display(
    priority_df[
        ["skill", "requirement_level", "skill_demand_pct",
         "role_priority_score"]
    ]
    .sort_values(
        ["skill_demand_pct", "role_priority_score"],
        ascending=False
    )
    .head(15)
)

Target role: Data Architect

Requirement levels in target role:
requirement_level
Rare          30
Supporting    20
Common         9
Core           2
Name: count, dtype: int64

Top 15 missing skills by role demand:


,skill,requirement_level,skill_demand_pct,role_priority_score
4,etl,Core,53.571429,0.388060
0,communication,Common,39.285714,0.672354
6,snowflake,Common,39.285714,0.274542
7,data warehousing,Common,35.714286,0.245636
10,databricks,Common,28.571429,0.215375
1,agile,Common,25.000000,0.452840
2,collaboration,Common,25.000000,0.422365
3,leadership,Common,25.000000,0.400360
11,data pipelines,Common,25.000000,0.214899
14,power bi,Supporting,17.857143,0.178434


In [16]:
from pathlib import Path

DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Save the personalized roadmap
roadmap_path = DATA_DIR / "learning_roadmap.csv"
learning_roadmap.to_csv(roadmap_path, index=False)

# Save the role-specific skill ranking
priority_path = DATA_DIR / "role_skill_priorities.csv"
priority_df.to_csv(priority_path, index=False)

print("Saved successfully:")
print(f"Learning roadmap: {roadmap_path.resolve()}")
print(f"Skill priorities: {priority_path.resolve()}")

Saved successfully:
Learning roadmap: C:\Users\murli mishra\OneDrive\Desktop\PYTHON\PathForgeAI\data\learning_roadmap.csv
Skill priorities: C:\Users\murli mishra\OneDrive\Desktop\PYTHON\PathForgeAI\data\role_skill_priorities.csv
